---
execute:
  enabled: false
---

# Lab: CNN Concepts from Arrays to Feature Maps {#ch-lab-cnn-concepts .unnumbered}

**Theory connection:** [Convolutional Neural Networks](../../chapters/09a-convolutional-neural-networks.qmd)

This lab makes the mechanics visible before a framework hides them. Most of the
work uses only NumPy. Complete each prediction or hand calculation before running
its code cell.

Download the [Jupyter notebook](../notebooks/cnn-concepts.ipynb) to run the activity.

## 1. Cross-Correlation by Hand {#sec-lab-cnn-cross-correlation}

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

image = np.array([
    [1, 1, 0, 0],
    [1, 1, 0, 0],
    [1, 1, 0, 0],
    [1, 1, 0, 0],
], dtype=float)

vertical_boundary = np.array([
    [1, -1],
    [1, -1],
], dtype=float)

Write all four products for the upper-left window and for the window one step to
its right. Predict the complete $3\times3$ result.

In [ ]:
upper_left = image[0:2, 0:2]
one_step_right = image[0:2, 1:3]
print("upper-left products:\n", upper_left * vertical_boundary)
print("upper-left sum:", np.sum(upper_left * vertical_boundary))
right_products = one_step_right * vertical_boundary
print("one-step-right products:\n", right_products)
print("one-step-right sum:", np.sum(right_products))

## 2. Implement the General Operation {#sec-lab-cnn-implementation}

In [ ]:
def cross_correlate2d(image, kernel, stride=1, padding=0):
    """2D cross-correlation for one image channel and one kernel."""
    image = np.asarray(image, dtype=float)
    kernel = np.asarray(kernel, dtype=float)
    if image.ndim != 2 or kernel.ndim != 2:
        raise ValueError(
            "image and kernel must both be two-dimensional"
        )
    if stride < 1 or padding < 0:
        raise ValueError(
            "stride must be positive and padding nonnegative"
        )

    padded = np.pad(image, padding)
    kh, kw = kernel.shape
    out_h = (padded.shape[0] - kh) // stride + 1
    out_w = (padded.shape[1] - kw) // stride + 1
    if out_h < 1 or out_w < 1:
        raise ValueError("kernel is larger than the padded input")

    output = np.empty((out_h, out_w), dtype=float)
    for i in range(out_h):
        for j in range(out_w):
            row = i * stride
            col = j * stride
            patch = padded[row:row + kh, col:col + kw]
            output[i, j] = np.sum(patch * kernel)
    return output

feature_map = cross_correlate2d(image, vertical_boundary)
expected = np.array([[0, 2, 0], [0, 2, 0], [0, 2, 0]], dtype=float)
np.testing.assert_array_equal(feature_map, expected)
feature_map

Why is this formally cross-correlation rather than strict mathematical
convolution? Confirm your answer by flipping the kernel on both axes and comparing
the output.

In [ ]:
flipped_kernel = np.flip(vertical_boundary)
strict_convolution = cross_correlate2d(image, flipped_kernel)
print(strict_convolution)

## 3. Padding and Stride {#sec-lab-cnn-stride-padding}

In [ ]:
def output_size(input_size, kernel_size, stride=1, padding=0):
    return (input_size + 2 * padding - kernel_size) // stride + 1

cases = [
    (28, 3, 1, 0),
    (28, 3, 1, 1),
    (28, 3, 2, 1),
]
for n, k, s, p in cases:
    print((n, k, s, p), "->", output_size(n, k, s, p))

For each case, interpret the result before continuing. Then test the function
against actual arrays.

In [ ]:
dummy = np.zeros((28, 28))
kernel3 = np.ones((3, 3))
assert cross_correlate2d(dummy, kernel3).shape == (26, 26)
assert cross_correlate2d(dummy, kernel3, padding=1).shape == (28, 28)
stride_two = cross_correlate2d(
    dummy, kernel3, stride=2, padding=1
)
assert stride_two.shape == (14, 14)

Explain why stride 2 can lose information even when the output shape is convenient.

## 4. Pooling Is a Different Operation {#sec-lab-cnn-pooling}

In [ ]:
def max_pool2d(feature_map, pool_size=2, stride=2):
    feature_map = np.asarray(feature_map)
    out_h = (feature_map.shape[0] - pool_size) // stride + 1
    out_w = (feature_map.shape[1] - pool_size) // stride + 1
    output = np.empty((out_h, out_w), dtype=feature_map.dtype)
    for i in range(out_h):
        for j in range(out_w):
            row = i * stride
            col = j * stride
            patch = feature_map[
                row:row + pool_size,
                col:col + pool_size,
            ]
            output[i, j] = np.max(patch)
    return output

pool_input = np.array([
    [1, 3, 2, 0],
    [4, 6, 1, 2],
    [0, 1, 5, 3],
    [2, 2, 4, 8],
])
pooled = max_pool2d(pool_input)
np.testing.assert_array_equal(pooled, np.array([[6, 2], [2, 8]]))
pooled

List two differences between convolution and pooling. Your answer should address
learned parameters and the meaning of the output.

## 5. Multiple Input and Output Channels {#sec-lab-cnn-channels}

One filter spans every input channel but produces one output feature map. A bank
of filters produces a bank of output channels.

In [ ]:
def cross_correlate_channels(image, kernels, bias=None):
    """Cross-correlate HWC input with kh-kw-Cin-Cout kernels."""
    image = np.asarray(image, dtype=float)
    kernels = np.asarray(kernels, dtype=float)
    if image.ndim != 3 or kernels.ndim != 4:
        raise ValueError(
            "expected HWC input and kh-kw-Cin-Cout kernels"
        )
    kh, kw, c_in, c_out = kernels.shape
    if image.shape[2] != c_in:
        raise ValueError(
            "input channels and kernel depth do not match"
        )
    if bias is None:
        bias = np.zeros(c_out)

    outputs = []
    for d in range(c_out):
        channel_sum = None
        for c in range(c_in):
            response = cross_correlate2d(
                image[:, :, c],
                kernels[:, :, c, d],
            )
            channel_sum = (
                response
                if channel_sum is None
                else channel_sum + response
            )
        outputs.append(channel_sum + bias[d])
    return np.stack(outputs, axis=-1)

two_channel_image = np.stack([image, 1 - image], axis=-1)
kernel_bank = np.zeros((2, 2, 2, 2))
# Filter 0 reads channel 0; filter 1 reads channel 1.
kernel_bank[:, :, 0, 0] = vertical_boundary
kernel_bank[:, :, 1, 1] = -vertical_boundary

multi_output = cross_correlate_channels(
    two_channel_image, kernel_bank
)
print("input shape:", two_channel_image.shape)
print("kernel-bank shape:", kernel_bank.shape)
print("output shape:", multi_output.shape)

Change the number of filters from two to three. Which dimension changes? Then
change the number of input channels. Which kernel dimension must also change?

## 6. Count Parameters Before Asking a Framework {#sec-lab-cnn-parameters}

In [ ]:
def conv_parameter_count(kh, kw, c_in, c_out, use_bias=True):
    return kh * kw * c_in * c_out + (c_out if use_bias else 0)

assert conv_parameter_count(3, 3, 1, 16) == 160
assert conv_parameter_count(3, 3, 16, 32) == 4640
print(conv_parameter_count(3, 3, 16, 32))

Calculate both answers by hand. Why does height or width not appear in the
parameter formula? Contrast this with the number of multiplications performed.

## 7. Optional Keras Check {#sec-lab-cnn-framework-check}

This final check requires Keras with a TensorFlow backend. The conceptual work
above remains fully runnable with NumPy alone.

In [ ]:
import keras
from keras import layers

shape_model = keras.Sequential([
    keras.Input(shape=(28, 28, 1)),
    layers.Conv2D(16, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(2),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(2),
], name="shape_check")
shape_model.summary()
assert shape_model.count_params() == 4800

Annotate each summary row with the output-shape calculation and parameter-count
calculation that produced it.

## Reflection {#sec-lab-cnn-concepts-reflection}

1. Why can one small kernel detect a pattern in several locations?
2. Why does one filter produce one output channel even when the input has many channels?
3. What information can pooling remove?
4. Which calculation was hardest to predict without running code, and what rule now resolves it?